# PEARL: Public Reproducibility Demo

**Policy Evolution through Aligned Retrospective Learning**

This notebook demonstrates the complete PEARL pipeline on **synthetic data** that requires no proprietary data access. All steps reproduce the algorithmic structure described in the manuscript:

1. Synthetic population generation (calibrated to MEPS + Camden characteristics)
2. WPAD preference pair construction (Algorithm 1)
3. Intervention Misalignment Index (IMI) estimation
4. PEARL tabular DPO training (Algorithm 2; tabular proxy for the full LLM pipeline)
5. DR-OPE evaluation and comparator benchmark
6. Camden Coalition reanalysis simulation

**Runtime**: ~2 minutes on a standard laptop CPU (no GPU required).

**Note**: Results on synthetic data differ from manuscript results, which are obtained from proprietary ACO care management records. The synthetic generator is calibrated to produce realistic population statistics but does not reproduce manuscript effect sizes exactly.

## Setup

In [ ]:
import sys
import os

# Ensure project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Python {sys.version}")
print(f"NumPy {np.__version__}, pandas {pd.__version__}")

## 1. Generate Synthetic Population

The generator creates a rising-risk ACO patient population with:
- Heterogeneous chronic disease profiles (CHF, COPD, diabetes, hypertension, CKD)
- Social determinants of health (ADI, food insecurity, housing instability)
- Staggered ACO onboarding WPAD events (Type 1, primary identification)
- Known ground-truth intervention misalignment (set by data-generating process)

In [ ]:
from data.synthetic_generator import generate_synthetic_population

pop = generate_synthetic_population(
    n_patients=50_000,
    n_wpad_primary=4_000,
    seed=42,
    true_imi=0.38,
)

patients = pop.patients
wpad_pairs = pop.wpad_pairs

print(f"Patients:               {len(patients):,}")
print(f"WPAD pairs (Type 1):    {len(wpad_pairs):,}")
print(f"Ground-truth IMI:       {pop.ground_truth_imi:.1%}")
print()
print("Intervention distribution (behavioral policy):")
print(patients["behavioral_intervention"].value_counts().to_string())

In [ ]:
# Patient demographics summary
summary = pd.DataFrame({
    "Mean": patients[["age", "charlson_score", "prior_ed_visits_6mo", "adi_percentile"]].mean(),
    "SD":   patients[["age", "charlson_score", "prior_ed_visits_6mo", "adi_percentile"]].std(),
}).round(1)
summary.index.name = "Feature"
print("Patient characteristics (synthetic population):")
print(summary.to_string())

binary_cols = ["female", "has_diabetes", "has_chf", "has_copd",
               "food_insecure", "housing_unstable"]
print()
print("Binary features (%):")
print((patients[binary_cols].mean() * 100).round(1).to_string())

## 2. WPAD Preference Pair Structure (Algorithm 1)

Each WPAD pair contains:
- `x`: patient feature vector (12-month covariate context)
- `y_preferred`: intervention type in the ON-window when outcome was good (Y=0)
- `y_rejected`: intervention type in the OFF-window when outcome was bad (Y=1)
- `iptw_weight`: inverse propensity weight for cross-patient supplementary pairs
- `direction`: +1 if OFF follows ON, −1 if OFF precedes ON (T6 direction test)

In [ ]:
print("WPAD pair columns:")
print(list(wpad_pairs.columns))
print()
print(f"Preferred intervention distribution:")
print(wpad_pairs["y_preferred"].value_counts().to_string())
print()
print(f"Rejected intervention distribution:")
print(wpad_pairs["y_rejected"].value_counts().to_string())
print()
print(f"IPTW weight range: [{wpad_pairs['iptw_weight'].min():.2f}, {wpad_pairs['iptw_weight'].max():.2f}]")
print(f"Direction (OFF before ON = -1): {(wpad_pairs['direction'] == -1).mean():.1%}")

## 3. Falsification Tests (T1–T5 Summary)

Pre-specified tests of the administrative exogeneity assumption.

In [ ]:
from evaluation.falsification_tests import FalsificationTests

ft = FalsificationTests(seed=42)
results = ft.run_all(wpad_pairs)

print("Falsification test results:")
for name, res in results.items():
    status = res.get("status", "?")
    detail = res.get("detail", "")
    print(f"  {name:6s}: {status:10s}  {detail}")

## 4. IMI Estimation (Behavioral Policy)

The Intervention Misalignment Index measures the fraction of patients receiving a suboptimal intervention under the current behavioral routing policy.

In [ ]:
from models.imi_estimator import IMIEstimator

# Use a test split for evaluation
rng = np.random.default_rng(0)
test_mask = rng.random(len(patients)) < 0.20
patients_test = patients[test_mask].reset_index(drop=True)
patients_train = patients[~test_mask].reset_index(drop=True)

imi_est = IMIEstimator(seed=42)
imi_est.fit(patients_train, wpad_pairs)

imi_result = imi_est.estimate(patients_test)

print(f"Estimated IMI (behavioral policy):  {imi_result['imi']:.1%}",
      f"(95% CI, {imi_result['ci_lower']:.1%}–{imi_result['ci_upper']:.1%})")
print(f"Ground-truth IMI (synthetic):       {pop.ground_truth_imi:.1%}")
print()
print("IMI by ADI quintile (equity-IMI analysis):")
for q, val in imi_result["imi_by_adi_quintile"].items():
    print(f"  Q{q}: {val:.1%}")

## 5. PEARL Training (Algorithm 2 — Tabular DPO Proxy)

This demonstrates the full algorithmic structure:
- Demographic-stratified IPTW upsampling
- Per-group DPO loss with equal group weights
- Abstention mechanism (DPO margin < τ → defer to standard routing)

The tabular logistic regression proxy captures all algorithmic structure without requiring GPU resources. The full LLM pipeline (Llama-3.1-8B) uses the same Algorithm 2 structure.

In [ ]:
from models.pearl_dpo import TabularPEARL

pearl = TabularPEARL(
    beta=0.10,
    abstention_threshold=0.30,
    group_equal_weights=True,
    seed=42,
)

train_history = pearl.fit(
    patients_train,
    wpad_pairs,
    n_iterations=80,
    verbose=True,
)

print(f"\nTraining complete. Final DPO loss: {train_history['final_loss']:.4f}")

In [ ]:
# Training convergence plot
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(train_history["loss_per_iter"], color="steelblue", lw=1.5)
ax.set_xlabel("Training iteration")
ax.set_ylabel("Group-stratified DPO loss")
ax.set_title("PEARL training convergence (synthetic data)")
ax.axhline(train_history["final_loss"], color="gray", lw=0.8, ls="--")
plt.tight_layout()
plt.savefig("../outputs/demo_training_curve.pdf", bbox_inches="tight")
plt.show()
print("Saved: outputs/demo_training_curve.pdf")

## 6. DR-OPE Evaluation

Doubly robust off-policy evaluation using the Marginalized DR estimator.
Lower DR-OPE = fewer predicted acute care events = better policy.

In [ ]:
from evaluation.drope_evaluator import DROPEEvaluator
from models.comparators import (
    LACEIndexPolicy, HOSPITALScorePolicy, XGBoostPolicy,
    BehavioralCloningPolicy, ObservationalDPOPolicy,
    CausalForestPolicy, DecisionTransformerPolicy, CQLPolicy,
)

drope_eval = DROPEEvaluator(n_bootstrap=1000, seed=42)

# Define policies for evaluation
policies = {
    "PEARL (MoE Router)": lambda pts: pearl.predict(pts),
    "Behavioral Policy":  lambda pts: pts["behavioral_intervention"].values,
}

# Add comparators (train on training set, evaluate on test set)
comparators = {
    "LACE Index (C1)": LACEIndexPolicy(),
    "HOSPITAL Score (C2)": HOSPITALScorePolicy(),
    "XGBoost (C3)": XGBoostPolicy(),
    "Behavioral Cloning SFT (C4)": BehavioralCloningPolicy(),
    "Observational DPO (C5)": ObservationalDPOPolicy(),
    "Causal Forest CATE (C6)": CausalForestPolicy(),
    "Decision Transformer (C7)": DecisionTransformerPolicy(),
    "CQL Offline RL (C8)": CQLPolicy(),
}

for name, comp in comparators.items():
    comp.fit(patients_train, wpad_pairs)
    policies[name] = lambda pts, c=comp: c.predict(pts)

print("Comparators fitted.")

In [ ]:
comparison_df = drope_eval.compare_policies(patients_test, policies)

print("DR-OPE Policy Comparison (lower = better):")
print(comparison_df.to_string(index=False))

In [ ]:
# Forest plot of DR-OPE values
fig, ax = plt.subplots(figsize=(7, 5))

df_plot = comparison_df.sort_values("dr_ope_mean", ascending=True).reset_index(drop=True)
colors = ["#1a5276" if n == "PEARL (MoE Router)" else
          "#922b21" if n == "Behavioral Policy" else "#555"
          for n in df_plot["policy"]]

ax.barh(
    df_plot["policy"], df_plot["dr_ope_mean"],
    xerr=[df_plot["dr_ope_mean"] - df_plot["ci_lower"],
          df_plot["ci_upper"] - df_plot["dr_ope_mean"]],
    color=colors, height=0.6, capsize=3, error_kw={"lw": 1.2}
)
ax.set_xlabel("DR-OPE policy value (lower = fewer acute care events)")
ax.set_title("Policy comparison — synthetic data demo")
ax.axvline(
    comparison_df.loc[comparison_df["policy"] == "Behavioral Policy", "dr_ope_mean"].values[0],
    color="#922b21", lw=1, ls=":"
)
plt.tight_layout()
plt.savefig("../outputs/demo_drope_forest.pdf", bbox_inches="tight")
plt.show()
print("Saved: outputs/demo_drope_forest.pdf")

In [ ]:
# Paired bootstrap comparison: PEARL vs. Behavioral
paired_test = drope_eval.paired_bootstrap_comparison(
    patients=patients_test,
    policy_fn_a=policies["PEARL (MoE Router)"],
    policy_fn_b=policies["Behavioral Policy"],
    name_a="PEARL (MoE Router)",
    name_b="Behavioral Policy",
)

print("Paired bootstrap comparison (PEARL vs. Behavioral Policy):")
print(f"  DR-OPE difference (Behavioral − PEARL): {paired_test['diff_point']:+.4f}")
print(f"  95% CI: [{paired_test['diff_ci_lower']:+.4f}, {paired_test['diff_ci_upper']:+.4f}]")
print(f"  One-sided p (PEARL ≥ Behavioral): {paired_test['p_value_one_sided']:.4f}")

## 7. IMI Reduction: PEARL vs. Behavioral Policy

In [ ]:
# IMI under PEARL
pearl_actions = pearl.predict(patients_test)
patients_test_pearl = patients_test.copy()
patients_test_pearl["behavioral_intervention"] = pearl_actions

imi_pearl = imi_est.estimate(patients_test_pearl)

imi_behavioral_val = imi_result["imi"]
imi_pearl_val = imi_pearl["imi"]
relative_reduction = (imi_behavioral_val - imi_pearl_val) / imi_behavioral_val

print(f"IMI under behavioral policy: {imi_behavioral_val:.1%}",
      f"(95% CI, {imi_result['ci_lower']:.1%}–{imi_result['ci_upper']:.1%})")
print(f"IMI under PEARL:             {imi_pearl_val:.1%}",
      f"(95% CI, {imi_pearl['ci_lower']:.1%}–{imi_pearl['ci_upper']:.1%})")
print(f"Relative IMI reduction:      {relative_reduction:.1%}")
print()
print("IMI by ADI quintile — PEARL vs. Behavioral:")
print(f"{'Quintile':>8}  {'Behavioral':>12}  {'PEARL':>8}")
for q in range(1, 6):
    beh = imi_result["imi_by_adi_quintile"].get(q, float("nan"))
    prl = imi_pearl["imi_by_adi_quintile"].get(q, float("nan"))
    print(f"{q:>8}  {beh:>12.1%}  {prl:>8.1%}")

## 8. Camden Coalition Reanalysis (Synthetic)

Applies PEARL to patients whose covariate profiles match the Camden Coalition trial population
(Finkelstein et al., *NEJM* 2019). Under a uniform protocol (all patients receive intensive
multidisciplinary care), IMI is estimated for the Camden-profile stratum.

In [ ]:
camden_patients = pop.camden_stratum_patients.copy()
print(f"Camden-stratum patients (matched profiles): {len(camden_patients):,}")

# Under Camden protocol: all patients receive intensive multidisciplinary care
camden_patients["behavioral_intervention"] = "clinical_complexity"
imi_camden_protocol = imi_est.estimate(camden_patients)

# Under PEARL: personalized recommendations
pearl_recs_camden = pearl.predict(camden_patients)
camden_patients_pearl = camden_patients.copy()
camden_patients_pearl["behavioral_intervention"] = pearl_recs_camden
imi_camden_pearl = imi_est.estimate(camden_patients_pearl)

print()
print("Camden protocol (uniform intensive):")
print(f"  IMI = {imi_camden_protocol['imi']:.1%}",
      f"(95% CI, {imi_camden_protocol['ci_lower']:.1%}–{imi_camden_protocol['ci_upper']:.1%})")
print()
print("PEARL (personalized):")
print(f"  IMI = {imi_camden_pearl['imi']:.1%}",
      f"(95% CI, {imi_camden_pearl['ci_lower']:.1%}–{imi_camden_pearl['ci_upper']:.1%})")
print()
print("PEARL intervention distribution for Camden-profile patients:")
print(pd.Series(pearl_recs_camden).value_counts(normalize=True).round(3).to_string())
print()
print("Interpretation: Among patients whose clinical profiles match the Camden trial,")
print(f"the uniform intensive protocol produces an estimated IMI of",
      f"{imi_camden_protocol['imi']:.1%} — i.e., that fraction would benefit more from")
print("a different intervention type than intensive multidisciplinary care.")

## 9. Summary Output Table

In [ ]:
pearl_row = comparison_df[comparison_df["policy"] == "PEARL (MoE Router)"].iloc[0]
beh_row   = comparison_df[comparison_df["policy"] == "Behavioral Policy"].iloc[0]

summary = {
    "N patients (test set)": len(patients_test),
    "WPAD pairs (training)": len(wpad_pairs),
    "ESS": f"{drope_eval.last_ess:.0f}",
    "IMI — behavioral policy": f"{imi_behavioral_val:.1%} (95% CI {imi_result['ci_lower']:.1%}–{imi_result['ci_upper']:.1%})",
    "IMI — PEARL": f"{imi_pearl_val:.1%} (95% CI {imi_pearl['ci_lower']:.1%}–{imi_pearl['ci_upper']:.1%})",
    "IMI relative reduction": f"{relative_reduction:.1%}",
    "DR-OPE — PEARL": f"{pearl_row['dr_ope_mean']:.3f} (95% CI {pearl_row['ci_lower']:.3f}–{pearl_row['ci_upper']:.3f})",
    "DR-OPE — Behavioral": f"{beh_row['dr_ope_mean']:.3f} (95% CI {beh_row['ci_lower']:.3f}–{beh_row['ci_upper']:.3f})",
    "DR-OPE difference (Behavioral − PEARL)": f"{paired_test['diff_point']:+.4f} (95% CI {paired_test['diff_ci_lower']:+.4f}–{paired_test['diff_ci_upper']:+.4f})",
    "One-sided p (PEARL ≥ Behavioral)": f"{paired_test['p_value_one_sided']:.4f}",
    "PEARL rank (of 10 policies)": int(comparison_df[comparison_df["policy"] == "PEARL (MoE Router)"].index[0] + 1),
}

print("Demo results summary:")
print("-" * 70)
for k, v in summary.items():
    print(f"  {k:<45} {v}")
print("-" * 70)
print()
print("Note: These are synthetic data results. Manuscript results are derived")
print("from proprietary ACO care management records and differ in magnitude.")

## References

Full reference list in the manuscript. Key citations for this notebook:

- Finkelstein A, et al. Health care hotspotting — a randomized, controlled trial. *NEJM*. 2020;382(2):152–162.
- Rafailov R, et al. Direct preference optimization: your language model is secretly a reward model. *NeurIPS*. 2023;36.
- Kallus N, Uehara M. Doubly robust off-policy value and gradient estimation for deterministic policies. *NeurIPS*. 2020;33.
- Sagawa S, et al. Distributionally robust neural networks for group shifts. *ICLR*. 2020.
- Wager S, Athey S. Estimation and inference of heterogeneous treatment effects using random forests. *J Am Stat Assoc*. 2018;113(523):1228–1242.
- Kumar A, et al. Conservative Q-learning for offline reinforcement learning. *NeurIPS*. 2020;33.
- Patel SY, Baum A, Basu S. Prediction of non-emergent acute care utilization and cost among patients receiving Medicaid. *Sci Rep*. 2024;14:824.